# Genie test
I test genie by having a question for genie. Writing my own SQL code for the query and comparing it with Genie's response.

## Test of mart_time

#### Question: What is the distribution of events by event country?


My code

In [0]:
%sql
SELECT
    event_country,
    COUNT(DISTINCT event) AS Events
FROM marathos.gold.mart_time
GROUP BY event_country
ORDER BY Events DESC;

Genie Code

In [0]:
%sql
SELECT
  `event_country`,
  COUNT(DISTINCT `event`) as event_count
FROM
  `marathos`.`gold`.`mart_time`
WHERE
  `event_country` IS NOT NULL
GROUP BY
  `event_country`
ORDER BY
  event_count DESC

#### Question: What is the distribution (min, max, average, median) of athlete ages?

My code

In [0]:
%sql
SELECT
    MIN(athlete_age) AS min_age,
    MAX(athlete_age) AS max_age,
    MEAN(athlete_age) AS mean_age,
    AVG(athlete_age) AS average_age
FROM marathos.gold.mart_time;

Genie Code

In [0]:
%sql
SELECT
  MIN(`athlete_age`) AS min_age,
  MAX(`athlete_age`) AS max_age,
  AVG(`athlete_age`) AS avg_age,
  PERCENTILE_APPROX(`athlete_age`, 0.5) AS median_age
FROM
  `marathos`.`gold`.`mart_time`
WHERE
  `athlete_age` IS NOT NULL;

#### Question: What is the longest distance an athlete have run on a 24 hour race?

My Code

In [0]:
%sql
SELECT
    MAX(athlete_distance_km)
FROM marathos.gold.mart_time
WHERE event_time_h = 24;

Genie Code

In [0]:
%sql
SELECT
  `athlete_distance_km`,
  `event`,
  `event_country`,
  `date`,
  `athlete_gender`,
  `athlete_country`,
  `athlete_age`,
  `event_time_h`,
  ROUND(`athlete_distance_km` / `event_time_h`, 2) AS `speed_km_h`
FROM
  `marathos`.`gold`.`mart_time`
WHERE
  `event_time_h` = 24
ORDER BY
  `athlete_distance_km` DESC
LIMIT 10

## Test of mart_distance

#### Question: What is the best time to run 100 km?

My Code

In [0]:
%sql
SELECT
    MIN(athlete_time_h) AS best_time
FROM marathos.gold.mart_distance
WHERE event_distance = 100;

Genie Code

In [0]:
%sql
SELECT
  `athlete_gender`,
  MIN(`athlete_time_h`) as best_time_h
FROM
  `marathos`.`gold`.`mart_distance`
WHERE
  `event_distance` = 100
  AND `athlete_time_h` IS NOT NULL
  AND `athlete_gender` IN ('M', 'F')
GROUP BY
  `athlete_gender`
ORDER BY
  best_time_h ASC

#### Question: How much faster are men than women at running 50 km year by year?

My Code

In [0]:
%sql
SELECT
    event_year,
    ROUND(AVG(athlete_speed_km_h) FILTER (WHERE athlete_gender = 'M'),2) AS avg_male_speed,
    ROUND(AVG(athlete_speed_km_h) FILTER (WHERE athlete_gender = 'F'),2) AS avg_female_speed,
    ROUND(avg_male_speed-avg_female_speed,2) AS Differens
FROM marathos.gold.mart_distance
WHERE event_distance = 50
GROUP BY event_year
HAVING AVG(athlete_speed_km_h) FILTER (WHERE athlete_gender = 'F') IS NOT NULL
ORDER BY event_year;

Genie Code

In [0]:
%sql
SELECT
  `event_year`,
  ROUND(
    AVG(
      CASE
        WHEN `athlete_gender` = 'M' THEN `athlete_speed_km_h`
      END
    ),
    2
  ) as male_speed,
  ROUND(
    AVG(
      CASE
        WHEN `athlete_gender` = 'F' THEN `athlete_speed_km_h`
      END
    ),
    2
  ) as female_speed,
  ROUND(
    AVG(
      CASE
        WHEN `athlete_gender` = 'M' THEN `athlete_speed_km_h`
      END
    )
      - AVG(
        CASE
          WHEN `athlete_gender` = 'F' THEN `athlete_speed_km_h`
        END
      ),
    2
  ) as speed_difference
FROM
  `marathos`.`gold`.`mart_distance`
WHERE
  `event_distance` = 50
  AND `athlete_gender` IN ('M', 'F')
  AND `athlete_speed_km_h` IS NOT NULL
GROUP BY
  `event_year`
HAVING
  AVG(
    CASE
      WHEN `athlete_gender` = 'F' THEN `athlete_speed_km_h`
    END
  ) IS NOT NULL
ORDER BY
  `event_year`

#### Question: When did the first woman enter a 50 km marathon?

My Code

In [0]:
%sql
SELECT
    event_year,
    COUNT_IF(athlete_gender = 'F') AS nr_female
FROM marathos.gold.mart_distance
WHERE event_distance = 50 AND athlete_gender = "F"
GROUP BY event_year
ORDER BY event_year;

Genie Code

In [0]:
%sql
SELECT
  `event_year`,
  `date`,
  `event`,
  `event_country`,
  `athlete_country`,
  `athlete_age`,
  `athlete_time_h`,
  `athlete_speed_km_h`
FROM
  `marathos`.`gold`.`mart_distance`
WHERE
  `event_distance` = 50
  AND `athlete_gender` = 'F'
  AND `athlete_speed_km_h` IS NOT NULL
ORDER BY
  `event_year`,
  `date`
LIMIT 10